In [2]:
import json
import pandas as pd
import numpy as np

data = pd.read_excel("overall-predictions.xlsx")
data

,User Story,Oracle Domain,Oracle Tasks & Sensitive Features,Oracle Unique Sensitive Features,ReFair Domain,ReFair Tasks & Sensitive Features,ReFair Unique Sensitive Features
0,A bioinformatics company is using statistical ...,Biology,"{'classification': [], 'graph mining': []}",{},Biology,{},{}
1,A bioinformatics company is using text generat...,Biology,{'graph mining': []},{},Biology,{},{}
2,A bioinformatics company is using visual quest...,Biology,{'classification': []},{},Biology,{'classification': []},{}
3,A bioinformatics researcher is developing a sy...,Biology,{'graph mining': []},{},Biology,{'graph mining': []},{}
4,A bioinformatics researcher is interested in i...,Biology,{},{},Biology,{},{}
...,...,...,...,...,...,...,...
12396,I am a dermatologist who needs a tool to summa...,Dermatology,{},{},Dermatology,{},{}
12397,I work in sports journalism and need a tool to...,Sport,{},{},Sport,{},{}
12398,I work in the movie industry and I need a mach...,Movies,"{'data summarization': ['age', 'gender']}","{'age', 'gender'}",Movies,"{'data summarization': ['age', 'gender']}","{'age', 'gender'}"
12399,Researchers are using bigram models to analyze...,Biology,{},{},Biology,{},{}


In [3]:
wrong_predictions = pd.DataFrame(columns=["User Story","Oracle Domain", "ReFair Domain", "Oracle Tasks", "ReFair Tasks", "Oracle Features", "ReFair Features", "wrong", "oracle","refair", "percentage"])

for us in data.iterrows():
    oracle = set(eval(us[1]["Oracle Unique Sensitive Features"]))
    refair = set(eval(us[1]["ReFair Unique Sensitive Features"]))
    

    if  not refair == oracle:
        wrong = len(refair.difference(oracle))
        oracle = len(oracle)
        refair = len(refair)
        if oracle == 0: 
            percentage = "all wrong (0)"
        
        else: percentage = wrong/oracle
    
        wrong_predictions.loc[len(wrong_predictions)] = [us[1]["User Story"], us[1]["Oracle Domain"], us[1]["ReFair Domain"], us[1]["Oracle Tasks & Sensitive Features"],  us[1]["ReFair Tasks & Sensitive Features"], us[1]["Oracle Unique Sensitive Features"],  us[1]["ReFair Unique Sensitive Features"], wrong, oracle,refair, percentage]

wrong_predictions

,User Story,Oracle Domain,ReFair Domain,Oracle Tasks,ReFair Tasks,Oracle Features,ReFair Features,wrong,oracle,refair,percentage
0,A film critic needs to review multiple movies ...,Movies,Biology,"{'data summarization': ['age', 'gender']}",{},"{'age', 'gender'}",{},0,2,0,0.0
1,A healthcare organization is using machine lea...,Health,Finance & Marketing,"{'classification': ['age', 'geography', 'sex',...","{'anomaly detection': ['age', 'gender', 'race'...","{'geography', 'ethnicity', 'gender', 'age', 's...","{'race', 'age', 'sex', 'gender', 'geography'}",1,5,5,0.2
2,A literary scholar wants to develop a bidirect...,Literature,Literature,{'ranking': ['gender']},"{'clustering': ['author', 'gender', 'textual r...",{'gender'},"{'author', 'textual references to people and t...",2,1,3,2.0
3,A literary scholar wants to use kohonen neural...,Literature,Literature,"{'clustering': ['author', 'gender', 'textual r...",{},"{'author', 'gender', 'textual references to pe...",{},0,3,0,0.0
4,A news organization is using machine learning ...,News,Finance & Marketing,"{'clustering': ['author', 'geography', 'textua...","{'anomaly detection': ['age', 'gender', 'race'...","{'author', 'geography', 'textual references to...","{'race', 'age', 'sex', 'gender', 'geography'}",4,3,5,1.333333
...,...,...,...,...,...,...,...,...,...,...,...
427,"As an information systems manager, I want to u...",Information Systems,Information Systems,{},"{'classification': ['age', 'geography', 'race'...",{},"{'ethnic group', 'race', 'age', 'skin tone', '...",11,0,11,all wrong (0)
428,"As an information systems professional, I want...",Information Systems,Information Systems,{},"{'classification': ['age', 'geography', 'race'...",{},"{'race', 'age', 'skin tone', 'sex', 'gender', ...",7,0,7,all wrong (0)
429,"As an information systems professional, I want...",Information Systems,Information Systems,"{'classification': ['age', 'geography', 'race'...",{},"{'ethnic group', 'geography', 'ownership', 'ge...",{},0,11,0,0.0
430,"As an information systems professional, I want...",Information Systems,Information Systems,{},"{'classification': ['age', 'geography', 'race'...",{},"{'ethnic group', 'race', 'age', 'skin tone', '...",11,0,11,all wrong (0)


In [5]:
oracle_zero = 0
void_predictions = 0
totally_wrong = 0
for row in wrong_predictions.iterrows():
    if row[1]['percentage'] == "all wrong (0)":
        oracle_zero = oracle_zero +1;
    elif int(row[1]['percentage']) == 0:
        void_predictions = void_predictions + 1;
    elif int(row[1]['percentage']) >= 1: 
        totally_wrong = totally_wrong + 1;
    else: print(row[1]['percentage'])

print("Feature suggestions totally wrong due void oracle: {} - {}".format(oracle_zero, oracle_zero/len(wrong_predictions)))
print("Features suggestions totally void rather than not void oracle {} - {}".format(void_predictions, void_predictions/len(wrong_predictions)))
print("Feature suggestion totally different from a not void oracle: {} - {}".format(totally_wrong, totally_wrong/len(wrong_predictions)))

Feature suggestions totally wrong due void oracle: 204 - 0.4722222222222222
Features suggestions totally void rather than not void oracle 217 - 0.5023148148148148
Feature suggestion totally different from a not void oracle: 11 - 0.02546296296296296


In [6]:
wrong_domains_prediction = pd.DataFrame(columns=["User Story", "Oracle Domain", "ReFair Domain", "Oracle Tasks", "ReFair Tasks", "Oracle Features", "Refair Features"])

wrong_domains = {}
for row in wrong_predictions.iterrows():
    if row[1]["ReFair Domain"] != row[1]["Oracle Domain"]:
        if wrong_domains.get(row[1]["ReFair Domain"]) is not None:
            wrong_domains.update({row[1]["ReFair Domain"]: wrong_domains.get(row[1]["ReFair Domain"]) + 1})
        else: wrong_domains.update({row[1]["ReFair Domain"]: 1})
        wrong_domains_prediction.loc[len(wrong_domains_prediction)] = [row[1]["User Story"], row[1]["Oracle Domain"], row[1]["ReFair Domain"], list(json.loads(row[1]["Oracle Tasks"].replace("\'", "\"")).keys()), list(json.loads(row[1]["ReFair Tasks"].replace("\'", "\"")).keys()), row[1]["Oracle Features"], row[1]["ReFair Features"]]

sorted_wrong_domains = sorted(wrong_domains.items(), key=lambda x:x[1])
print(sorted_wrong_domains)

[('Movies', 1), ('Computer Vision', 1), ('Linguistics', 1), ('Pharmacology', 1), ('Sociology', 1), ('Information Systems', 1), ('Literature', 2), ('Finance & Marketing', 3), ('Medicine', 3), ('Psychology', 3), ('Biology', 4), ('Health', 5)]


In [38]:
wrong_domains_prediction.to_excel("Misclassified_Domains_Details.xlsx")
wrong_domains_prediction

,User Story,Oracle Domain,ReFair Domain,Oracle Tasks,ReFair Tasks,Oracle Features,Refair Features
0,A film critic needs to review multiple movies ...,Movies,Biology,[data summarization],[],"{'age', 'gender'}",{}
1,A healthcare organization is using machine lea...,Health,Finance & Marketing,"[classification, regression, representation le...","[anomaly detection, classification, clustering...","{'geography', 'ethnicity', 'gender', 'age', 's...","{'race', 'age', 'sex', 'gender', 'geography'}"
2,A news organization is using machine learning ...,News,Finance & Marketing,"[clustering, bias detection in word embeddings]","[anomaly detection, classification, clustering...","{'author', 'geography', 'textual references to...","{'race', 'age', 'sex', 'gender', 'geography'}"
3,A news organization wants to develop a bidirec...,News,Biology,"[ranking, bias detection in word embeddings]",[classification],"{'news provider', 'geography', 'textual refere...",{}
4,A news organization wants to use bootstrap agg...,News,Movies,[],"[ranking, representation learning]",{},"{'age', 'gender'}"
5,"As a book club member, I want to use a chatbot...",Literature,Health,[],"[classification, regression]",{},"{'age', 'sex', 'gender', 'geography', 'ethnici..."
6,"As a computer vision researcher, I want to use...",Biology,Computer Vision,"[classification, graph mining]","[classification, representation learning]",{},"{'race', 'age', 'skin tone', 'skin color', 'di..."
7,"As a doctor, I want a machine augmented intell...",Medicine,Literature,[classification],[ranking],"{'ethnicity', 'gender', 'race', 'age', 'sex'}",{'gender'}
8,"As a healthcare provider, I want to use recomm...",Biology,Health,[graph mining],"[graph diffusion, matching, representation lea...",{},"{'age', 'sex', 'gender', 'geography', 'ethnici..."
9,"As a language learner, I want to use a chatbot...",Linguistics,Health,"[entity resolution, sentiment analysis, bias d...","[classification, regression]","{'geography', 'religion', 'dialect', 'gender',...","{'age', 'sex', 'gender', 'geography', 'ethnici..."


In [7]:
wrong_tasks_prediction = pd.DataFrame(columns=["User Story", "Oracle Domain", "ReFair Domain", "Oracle Tasks", "ReFair Tasks", "Oracle Features", "Refair Features"])

wrong_tasks = {}
wrong_story = False
i = 0 
for row in wrong_predictions.iterrows():
    i = i + 1 
    refair_tasks = json.loads(row[1]["ReFair Tasks"].replace("\'", "\""))

    oracle_tasks = json.loads(row[1]["Oracle Tasks"].replace("\'", "\""))

    for task in refair_tasks.keys():
        if task not in oracle_tasks.keys():
            if wrong_tasks.get(task) is not None:
                wrong_tasks.update({task: int(wrong_tasks.get(task)) + 1})

            else: wrong_tasks.update({task: 1})
            wrong_story = True
        
    
    if wrong_story: wrong_tasks_prediction.loc[len(wrong_tasks_prediction)] = [row[1]["User Story"], row[1]["Oracle Domain"], row[1]["ReFair Domain"], list(json.loads(row[1]["Oracle Tasks"].replace("\'", "\"")).keys()), list(json.loads(row[1]["ReFair Tasks"].replace("\'", "\"")).keys()), row[1]["Oracle Features"], row[1]["ReFair Features"]]
    wrong_story = False
sorted_wrong_tasks = sorted(wrong_tasks.items(), key=lambda x:x[1])
print(sorted_wrong_tasks)

[('task assignment', 1), ('graph mining', 1), ('risk assessment', 2), ('pricing', 3), ('graph augmentation', 4), ('spatio-temporal process learning', 4), ('graph diffusion', 6), ('entity resolution', 6), ('resource allocation', 8), ('subset selection', 11), ('speech recognition', 12), ('matching', 14), ('bias detection in language models', 15), ('machine translation', 16), ('anomaly detection', 20), ('data summarization', 20), ('sentiment analysis', 21), ('bias detection in word embeddings', 22), ('regression', 35), ('representation learning', 38), ('ranking', 46), ('clustering', 55), ('classification', 107)]


In [8]:
wrong_tasks_prediction.to_excel("Misclassified_Tasks_Details.xlsx")

wrong_tasks_prediction

,User Story,Oracle Domain,ReFair Domain,Oracle Tasks,ReFair Tasks,Oracle Features,Refair Features
0,A healthcare organization is using machine lea...,Health,Finance & Marketing,"[classification, regression, representation le...","[anomaly detection, classification, clustering...","{'geography', 'ethnicity', 'gender', 'age', 's...","{'race', 'age', 'sex', 'gender', 'geography'}"
1,A literary scholar wants to develop a bidirect...,Literature,Literature,[ranking],[clustering],{'gender'},"{'author', 'textual references to people and t..."
2,A news organization is using machine learning ...,News,Finance & Marketing,"[clustering, bias detection in word embeddings]","[anomaly detection, classification, clustering...","{'author', 'geography', 'textual references to...","{'race', 'age', 'sex', 'gender', 'geography'}"
3,A news organization wants to develop a bidirec...,News,Biology,"[ranking, bias detection in word embeddings]",[classification],"{'news provider', 'geography', 'textual refere...",{}
4,A news organization wants to use bootstrap agg...,News,Movies,[],"[ranking, representation learning]",{},"{'age', 'gender'}"
...,...,...,...,...,...,...,...
245,"As an information systems analyst, I want to u...",Information Systems,Information Systems,[],[representation learning],{},"{'race', 'age', 'skin tone', 'sex', 'gender', ..."
246,"As an information systems analyst, I want to u...",Information Systems,Information Systems,[],"[classification, data summarization, matching,...",{},"{'ethnic group', 'race', 'age', 'skin tone', '..."
247,"As an information systems manager, I want to u...",Information Systems,Information Systems,[],"[classification, data summarization, matching,...",{},"{'ethnic group', 'race', 'age', 'skin tone', '..."
248,"As an information systems professional, I want...",Information Systems,Information Systems,[],[classification],{},"{'race', 'age', 'skin tone', 'sex', 'gender', ..."
